# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp,regexp_replace
from pyspark.sql.types import *
load_dotenv()


# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

# Reusable Functions

In [0]:
def parse_mixed_date(df, col_name, output_col=None):
    out_col = output_col if output_col else col_name

    return df.withColumn(
        out_col,
        when(
            trim(col(col_name)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"),
            to_date(trim(col(col_name)), "dd-MM-yyyy")
        ).when(
            trim(col(col_name)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            to_date(trim(col(col_name)), "d/M/yyyy")
        ).otherwise(None)
    )

In [0]:
def parse_mixed_datetime(df, input_col, output_col=None):
    out_col = output_col if output_col else input_col
    return df.withColumn(
        out_col,
        when(
            trim(col(input_col)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}\s\d{1,2}:\d{2}$"),
            to_timestamp(trim(col(input_col)), "dd-MM-yyyy HH:mm")
        ).when(
            trim(col(input_col)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}\s\d{1,2}:\d{2}$"),
            to_timestamp(trim(col(input_col)), "d/M/yyyy H:mm")
        )
        .otherwise(None)
    )

# Date Cleaning

## Customer Table

In [0]:
customer_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_customer`""")

### Typecasting

In [0]:
customer_df = parse_mixed_date(customer_df, "account_created_date")

In [0]:
customer_df=customer_df.\
            withColumn("is_active",
            when(col("is_active")=="1",True)
            .when(col("is_active")=="0",False)
            .otherwise(None)
            )

In [0]:
customer_df=customer_df.withColumn("customer_id",col("customer_id").cast(IntegerType()))

### Trimming Spaces

In [0]:
customer_df=customer_df.withColumn("customer_name",trim(col("customer_name")))

## Employee Table

In [0]:
employee_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_employee`""")

### Typecasting

In [0]:
employee_df = parse_mixed_date(employee_df,"last_update")
employee_df = parse_mixed_date(employee_df,"hire_date")
employee_df = employee_df.withColumn("employee_id",col("employee_id").cast(IntegerType()))

In [0]:
employee_df=employee_df.\
            withColumn("is_active_flag",
            when(col("is_active_flag")=="Yes",True)
            .when(col("is_active_flag")=="No",False)
            .otherwise(None)
            )

### Trimming Spaces

In [0]:

employee_df=employee_df.withColumn("employee_name",trim(col("employee_name")))

## Product Table

In [0]:
product_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_product`""")

### Trimming Spaces

In [0]:
product_df=product_df.withColumn("product_name",trim(col("product_name")))

### Typecasting

In [0]:
product_df = product_df.\
            withColumn("is_active",
            when(col("is_active")=="Y",True)
            .when(col("is_active")=="N",False)
            .otherwise(None)
            )

In [0]:
product_df=parse_mixed_date(product_df,"created_date")

In [0]:
product_df=product_df.withColumns(
    {
        "list_price":col("list_price").cast(IntegerType()),
        "product_id":col("product_id").cast(IntegerType())
    }
    )

## Opportunity Table

In [0]:
opportunity_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_opportunity`""")

### Typecasting

In [0]:
opportunity_df=parse_mixed_datetime(opportunity_df,"created_timestamp")

In [0]:
opportunity_df=opportunity_df.withColumn("opportunity_id",col("opportunity_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("employee_id",col("employee_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("customer_id",col("customer_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("product_id",col("product_id").cast(IntegerType()))

### Removing Unnecessary Symbols

In [0]:
opportunity_df=opportunity_df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), r"[£$€,]", "").cast("double")
)

In [0]:
display(opportunity_df)